# Cloud recognition from frozen SEVIRI encoders

Questo esperimento verifica quanto le rappresentazioni apprese dai diversi obiettivi self-supervised permettano di riconoscere la copertura nuvolosa. Il target è il prodotto binario **`cma`** generato dal nuovo codice `CMA_original`: ciascun pixel valido è classificato come **sereno (0)** oppure **nuvoloso (1)**. Il valore 255 identifica un target mancante, escluso dalla loss e dalle metriche.

Il probe restituisce una probabilità di nube per ciascuno dei 224 × 224 pixel dell’ultimo frame osservato. L’encoder riceve i primi **10 frame**, distanziati di cinque minuti, della clip canonica di 16 frame; il target corrisponde al **frame 9**, cioè 45 minuti dopo l’inizio della clip. Il suo campo di input comprende tutti i sette canali SEVIRI, con la stessa normalizzazione globale usata nel rispettivo pretraining.

I punteggi misurano l’accordo con questa implementazione CMA su osservazioni di giornate distinte dal pretraining. Il confronto con un encoder casuale della stessa architettura quantifica il contributo del pretraining alla segmentazione. La verifica della corrispondenza con le nuvole atmosferiche reali richiederà anche osservazioni di riferimento indipendenti.

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src/emma_gpm').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from emma_gpm.cloud_probe_reporting import load_results, plot_comparison, plot_paired_gain, plot_examples

RESULTS = Path(os.environ.get('SEVIRI_CLOUD_PROBE_RESULTS', ROOT / 'results/cloud_mask_probe_20261031'))
DATASET = Path(os.environ.get('SEVIRI_CLOUD_PROBE_DATASET', '/data1/deep_convection_data/cloud_mask_probe_20261031'))
pd.options.display.float_format = '{:.3f}'.format
results = load_results(RESULTS)
print(f'Probe completati: {len(results)}')
if (RESULTS / 'progress.json').exists():
    status = json.loads((RESULTS / 'progress.json').read_text())
    print('Stato della serie:', status['status'])
    print('Modello in elaborazione:', status.get('current') or '—')
    print('Encoder ancora attesi:', len(status.get('pending', [])))

## Osservazioni e costruzione delle maschere

Il dataset iniziale comprende **448 clip di training**, tratte dal 13 e 14 dicembre 2020, **112 clip di validation**, del 16 dicembre, e **112 clip di test**, del 26 dicembre. Il programma controlla che queste giornate siano separate dalle date dei manifest di pretraining e validation di ogni encoder. Tra i gruppi del probe è presente almeno un’intera giornata inutilizzata.

Ogni istante selezionato fornisce 14 ritagli spaziali disgiunti. Gli inizi delle clip sono distanziati di 80 minuti nel training e di 160 minuti nella validation e nel test. Le clip della stessa giornata rappresentano strutture atmosferiche correlate; la valutazione iniziale dispone di una giornata di validation e una di test.

CMA elabora l’intero dominio disponibile di **763 × 1762 pixel**, procedendo cronologicamente ogni cinque minuti. La prima maschera selezionata segue un’ora di osservazioni; lo stato temporale dell’algoritmo viene inizializzato all’inizio di ciascuna giornata. La maschera del frame 9 viene conservata prima di elaborare i frame successivi. Il ritaglio avviene dopo la classificazione sul dominio completo.

La geometria satellitare deriva dalle coordinate dell’archivio. La distinzione terra–mare usa i poligoni Natural Earth alla scala 1:10 milioni, valutati nel centro del pixel. I riferimenti radiometrici superficiali e atmosferici sono stimati dall’implementazione Pure-IR a sette canali. Questa prima configurazione usa i valori predefiniti di quota pari a zero e assenza di un campo ausiliario di neve e ghiaccio: terreno montuoso e neve invernale sono quindi aspetti da considerare nell’interpretazione delle maschere.

Nel grafico seguente la frazione nuvolosa è la percentuale di pixel validi classificati come nube dal CMA, aggregata separatamente per ciascuno split.

In [ ]:
if (DATASET / 'manifest.json').exists():
    manifest = json.loads((DATASET / 'manifest.json').read_text())
    samples = manifest['samples']
    fractions = []
    for split in ['train', 'validation', 'test']:
        subset = [r for r in samples if r['split'] == split]
        fractions.append(100 * sum(r['cloud_pixels'] for r in subset) / sum(r['valid_pixels'] for r in subset))
    fig, ax = plt.subplots(figsize=(6, 3.5), layout='constrained')
    bars = ax.bar(['Training', 'Validation', 'Test'], fractions, color=['#367fb0', '#d28e3b', '#4a9a73'])
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylim(0, 100)
    ax.set_ylabel('Cloud-labelled pixels (%)')
    ax.set_title('CMA cloud fraction')
    plt.show()
else:
    print('Dataset in preparazione; il grafico apparirà al completamento.')

## Testa di segmentazione e selezione del modello

Tutti i pesi dell’encoder sono congelati. Le rappresentazioni vengono estratte una sola volta, in modalità evaluation, e salvate su disco. Il programma verifica mediante checksum che pesi e buffer dell’encoder restino identici durante l’estrazione. Il calcolo usa BF16 sulla CPU e conserva le rappresentazioni in array float32; i training self-supervised continuano sulla GPU.

Si usano i token dell’ultimo intervallo temporale dell’encoder, disposti su una griglia di **14 × 14 posizioni**. Attraverso l’attenzione dell’encoder, ciascun token contiene informazione proveniente dalla storia osservata. La testa applica LayerNorm, un layer Linear con 128 unità, GELU e un secondo Linear con 256 uscite. Le 256 uscite descrivono i 16 × 16 pixel della patch; assemblando tutte le patch si ottiene la mappa di probabilità a 224 × 224 pixel.

Si addestra soltanto questa testa per **30 epoche**, con batch size 16, AdamW, learning rate 0,001 e weight decay 0,0001. La loss è la binary cross-entropy sui pixel con target valido. I diversi encoder condividono gli split, il seed della testa e l’ordine dei minibatch. L’inizializzazione della testa è identica tra encoder della stessa architettura; Small e Base hanno dimensioni di ingresso differenti.

Per ciascuna combinazione di architettura, normalizzazione e seed iniziale viene addestrata anche la testa sopra un **encoder casuale**, indicato con **Random**. Il suo confronto appaiato usa la stessa procedura del modello preaddestrato. La validation seleziona l’epoca con la maggiore media tra IoU della nube e IoU del sereno; in caso di parità si conserva la prima epoca. La soglia di classificazione della probabilità di nube è fissata a **0,5**. Il test viene valutato dopo questa selezione.

## Confronto dei modelli

**Cloud IoU** confronta l’area di sovrapposizione tra nube predetta e nube target con l’area complessivamente occupata dalle due maschere. **Cloud Dice** misura anch’esso la sovrapposizione spaziale, attribuendo peso doppio ai pixel condivisi nel confronto con la somma delle due aree. I valori mostrati derivano dai conteggi dei pixel validi aggregati sull’intero test; un valore più alto indica un accordo maggiore con CMA.

Le legende indicano l’obiettivo del pretraining: **MVM** ricostruisce parti mascherate della storia; **Future** predice i frame futuri; **Joint** combina ricostruzione mascherata e previsione; **Residual** predice la correzione radiometrica rispetto alla previsione ottenuta per advezione. **Random** identifica il controllo con encoder inizializzato casualmente. Il gruppo della barra Random indica il modello cui il controllo è appaiato; i pesi del suo encoder restano quelli dell’inizializzazione. Ogni gruppo sull’asse orizzontale specifica architettura, numero di clip di pretraining e numero di epoche. Small con 16.384 clip ha ricevuto 75 epoche; gli altri esperimenti elencati ne hanno ricevute 150.

In [ ]:
results = load_results(RESULTS)
plot_comparison(results)
plt.show()

Il grafico successivo mostra, per ciascun modello preaddestrato, la differenza di Cloud IoU rispetto al proprio controllo casuale. L’unità è il **punto percentuale**: valori positivi indicano un vantaggio del pretraining con la stessa procedura di addestramento della testa. I confronti appaiono quando entrambi i probe sono terminati.

In [ ]:
plot_paired_gain(results)
plt.show()

## Apprendimento della testa

Le curve seguenti riguardano il primo modello completato della serie e il suo controllo casuale. L’asse orizzontale riporta le epoche di training della testa; l’asse verticale riporta la binary cross-entropy. La curva di training descrive i minibatch usati per ottimizzare la testa; quella di validation descrive l’applicazione della testa a una giornata separata. La selezione del checkpoint usa la mean IoU di validation, definita sopra.

In [ ]:
selected_case = None
if not results.empty:
    available = results[results.kind == 'pretrained'].sort_values(['clips', 'case'])
    if len(available):
        selected_case = available.iloc[0]['case']
        info = json.loads((RESULTS / selected_case / 'comparison.json').read_text())
        fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
        for case, label in [(selected_case, 'Pretrained'), (info['paired_random_control'], 'Random')]:
            path = RESULTS / case / 'history.json'
            if path.exists():
                history = json.loads(path.read_text())
                epochs = [r['epoch'] for r in history]
                line, = ax.plot(epochs, [r['train_bce'] for r in history], label=f'{label}: training')
                ax.plot(epochs, [r['validation']['bce'] for r in history], '--', color=line.get_color(), label=f'{label}: validation')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Head training epoch'); ax.set_ylabel('Binary cross-entropy')
        ax.set_title('Cloud probe learning curves'); ax.legend(); ax.grid(alpha=.2)
        plt.show()
        print('Modello rappresentato:', selected_case)

## Esempi spaziali sul test

Gli esempi provengono da posizioni fissate nell’elenco ordinato delle clip di test. Le colonne mostrano IR 10.8 µm, il target CMA e le probabilità di nube del modello preaddestrato e del controllo casuale. Nel campo IR, la scala comune copre 210–300 K e i valori più freddi appaiono più chiari. Per target e probabilità, il bianco indica zero e il blu scuro uno. La maschera CMA descrive quindi due classi, mentre le predizioni mostrano il grado di probabilità assegnato alla nube. Date e posizioni dei ritagli sono riportate dopo la figura.

La testa decodifica separatamente i pixel di ciascuna patch. Questa struttura può produrre discontinuità lungo bordi distanziati di 16 pixel, visibili nelle mappe di probabilità. Il confronto mantiene la stessa testa per tutti gli encoder.

In [ ]:
if selected_case is not None:
    figure, example_metadata = plot_examples(RESULTS, selected_case)
    plt.show()
    for index, description in enumerate(example_metadata, 1):
        print(f'Esempio {index}: {description}')

## Portata del confronto

Questo primo esperimento valuta la decodificabilità della maschera CMA sulle rappresentazioni congelate. La copertura temporale del test, il seed unico e le approssimazioni del teacher definiscono l’estensione attuale dei risultati. Un’analisi successiva potrà aggiungere più giornate, seed, ablazioni temporali e spettrali e un encoder RGB adattato come controllo, per distinguere il contributo delle firme spettrali da quello della storia atmosferica.

I file `result.json` conservano anche precisione, richiamo, IoU del sereno, balanced accuracy e conteggi della matrice di confusione. `test_per_clip.json` conserva i risultati di ciascuna clip. Le metriche sono conteggi di pixel nella griglia proiettata del satellite.

Per verificare da SSH l’avanzamento e gli eventuali arresti:

```bash
python3 /home/isac/deep_convection/scripts/status_seviri_cloud_probes.py
```

La descrizione riproducibile del protocollo e i comandi di ripresa sono in `docs/SEVIRI_CLOUD_MASK_PROBE.md`.